<a href="https://colab.research.google.com/github/JGPannekoek/NeuroSpark/blob/main/project_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install spikingjelly

In [ ]:
!pip install mne


In [ ]:
import numpy as np
import mne
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from spikingjelly.activation_based import neuron, surrogate,layer


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

folder_path = '/content/drive/MyDrive/project'
files_in_folder = os.listdir(folder_path)
print(files_in_folder)

['A01T.gdf', 'A01E.gdf', 'A02T.gdf', 'A02E.gdf', 'A03T.gdf', 'A03E.gdf', 'A04T.gdf', 'A04E.gdf']


In [ ]:
raw = mne.io.read_raw_gdf('/content/drive/MyDrive/project/A01T.gdf', preload=True)

print(raw.info)
print("Sampling rate:", raw.info['sfreq'])
print("Number of channels:", len(raw.ch_names))
print("Channel names:", raw.ch_names)

Extracting GDF parameters from /content/drive/MyDrive/project/A01T.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG, EOG-left, EOG-central, EOG-right
Creating raw.info structure...
Reading 0 ... 672527  =      0.000 ...  2690.108 secs...


/usr/lib/python3.13/contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


<Info | 8 non-empty values
 bads: []
 ch_names: EEG-Fz, EEG-0, EEG-1, EEG-2, EEG-3, EEG-4, EEG-5, EEG-C3, EEG-6, ...
 chs: 25 EEG
 custom_ref_applied: False
 highpass: 0.5 Hz
 lowpass: 100.0 Hz
 meas_date: 2005-01-17 12:00:00 UTC
 nchan: 25
 projs: []
 sfreq: 250.0 Hz
 subject_info: <subject_info | his_id: A01, sex: 0, last_name: X, birthday: 1983-01-17>
>
Sampling rate: 250.0
Number of channels: 25
Channel names: ['EEG-Fz', 'EEG-0', 'EEG-1', 'EEG-2', 'EEG-3', 'EEG-4', 'EEG-5', 'EEG-C3', 'EEG-6', 'EEG-Cz', 'EEG-7', 'EEG-C4', 'EEG-8', 'EEG-9', 'EEG-10', 'EEG-11', 'EEG-12', 'EEG-13', 'EEG-14', 'EEG-Pz', 'EEG-15', 'EEG-16', 'EOG-left', 'EOG-central', 'EOG-right']


In [ ]:
events, event_id = mne.events_from_annotations(raw)
print("Event IDs found:", event_id)
print("Number of events:", len(events))
print("First 10 events:\n", events[:10])

Used Annotations descriptions: ['1023', '1072', '276', '277', '32766', '768', '769', '770', '771', '772']
Event IDs found: {'1023': 1, '1072': 2, '276': 3, '277': 4, '32766': 5, '768': 6, '769': 7, '770': 8, '771': 9, '772': 10}
Number of events: 603
First 10 events:
 [[    0     0     5]
 [    0     0     3]
 [29683     0     5]
 [29683     0     4]
 [49955     0     5]
 [49955     0     2]
 [91518     0     5]
 [91868     0     6]
 [92368     0    10]
 [93871     0     6]]


In [ ]:
motor_imagery_codes = ['769', '770', '771', '772']
event_id_filtered = {k: v for k, v in event_id.items() if k in motor_imagery_codes}
print("Filtered event IDs:", event_id_filtered)

epochs = mne.Epochs(raw, events, event_id=event_id_filtered,
                     tmin=0, tmax=4.0, baseline=None, preload=True)

print(epochs)
print("Epochs data shape:", epochs.get_data().shape)

Filtered event IDs: {'769': 7, '770': 8, '771': 9, '772': 10}
Not setting metadata
288 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 288 events and 1001 original time points ...
0 bad epochs dropped
<Epochs | 288 events (all good), 0 – 4 s (baseline off), ~55.0 MiB, data loaded,
 '769': 72
 '770': 72
 '771': 72
 '772': 72>
Epochs data shape: (288, 25, 1001)


In [ ]:
epochs_eeg = epochs.copy().pick_types(eeg=True, eog=False)
print("Channels after removing EOG:", len(epochs_eeg.ch_names))
print("Channel names:", epochs_eeg.ch_names)
print("New data shape:", epochs_eeg.get_data().shape)

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Channels after removing EOG: 25
Channel names: ['EEG-Fz', 'EEG-0', 'EEG-1', 'EEG-2', 'EEG-3', 'EEG-4', 'EEG-5', 'EEG-C3', 'EEG-6', 'EEG-Cz', 'EEG-7', 'EEG-C4', 'EEG-8', 'EEG-9', 'EEG-10', 'EEG-11', 'EEG-12', 'EEG-13', 'EEG-14', 'EEG-Pz', 'EEG-15', 'EEG-16', 'EOG-left', 'EOG-central', 'EOG-right']
New data shape: (288, 25, 1001)


In [ ]:
#remove EOG directly
eog_channels = ['EOG-left', 'EOG-central', 'EOG-right']
epochs_eeg = epochs.copy().drop_channels(eog_channels)

print("Channels after removing EOG:", len(epochs_eeg.ch_names))
print("Channel names:", epochs_eeg.ch_names)
print("New data shape:", epochs_eeg.get_data().shape)



Channels after removing EOG: 22
Channel names: ['EEG-Fz', 'EEG-0', 'EEG-1', 'EEG-2', 'EEG-3', 'EEG-4', 'EEG-5', 'EEG-C3', 'EEG-6', 'EEG-Cz', 'EEG-7', 'EEG-C4', 'EEG-8', 'EEG-9', 'EEG-10', 'EEG-11', 'EEG-12', 'EEG-13', 'EEG-14', 'EEG-Pz', 'EEG-15', 'EEG-16']
New data shape: (288, 22, 1001)


In [ ]:
# 1. Bandpass Filter: 7-35 Hz (cover mu وbeta rhythms)
epochs_filtered = epochs_eeg.copy().filter(l_freq=7.0, h_freq=35.0)

# 2. Common Average Reference (CAR)
epochs_car = epochs_filtered.copy().set_eeg_reference(ref_channels='average')

print("Data shape after filtering + CAR:", epochs_car.get_data().shape) #mu (8-12Hz) وbeta (13-30Hz) rhythms

Setting up band-pass filter from 7 - 35 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 7.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 6.00 Hz)
- Upper passband edge: 35.00 Hz
- Upper transition bandwidth: 8.75 Hz (-6 dB cutoff frequency: 39.38 Hz)
- Filter length: 413 samples (1.652 s)

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Data shape after filtering + CAR: (288, 22, 1001)


In [ ]:
data = epochs_car.get_data()  # shape: (288, 22, 1001)

# z-score normalization per trial, per channel
mean = data.mean(axis=2, keepdims=True)  # mean across time, for each (trial, channel)
std = data.std(axis=2, keepdims=True)
data_normalized = (data - mean) / (std + 1e-8)  # +1e-8 to avoid division by zero

print("Normalized data shape:", data_normalized.shape)
print("Mean after normalization (should be ~0):", data_normalized.mean())
print("Std after normalization (should be ~1):", data_normalized.std())


Normalized data shape: (288, 22, 1001)
Mean after normalization (should be ~0): -3.7047485489836573e-19
Std after normalization (should be ~1): 0.9960184214777902


In [ ]:
from sklearn.model_selection import train_test_split

labels_raw = epochs_car.events[:, -1]  # the actual event codes: 7, 8, 9, 10
print("Unique raw labels:", np.unique(labels_raw))

label_map = {7: 0, 8: 1, 9: 2, 10: 3}
labels = np.array([label_map[l] for l in labels_raw])
print("Unique mapped labels:", np.unique(labels))
print("Label distribution:", np.bincount(labels))

X_train, X_val, y_train, y_val = train_test_split(
    data_normalized, labels, test_size=0.2, stratify=labels, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("y_train distribution:", np.bincount(y_train))
print("y_val distribution:", np.bincount(y_val))

Unique raw labels: [ 7  8  9 10]
Unique mapped labels: [0 1 2 3]
Label distribution: [72 72 72 72]
X_train shape: (230, 22, 1001)
X_val shape: (58, 22, 1001)
y_train distribution: [57 58 57 58]
y_val distribution: [15 14 15 14]


In [ ]:
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.LongTensor(y_train)
X_val_tensor = torch.FloatTensor(X_val)
y_val_tensor = torch.LongTensor(y_val)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

for batch_x, batch_y in train_loader:
    print("Batch X shape:", batch_x.shape)
    print("Batch y shape:", batch_y.shape)
    break

Batch X shape: torch.Size([16, 22, 1001])
Batch y shape: torch.Size([16])


In [ ]:
class FeatureExtractionLayer(nn.Module):
    def __init__(self, in_channels=22, hidden_channels=16):
        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels=in_channels,
            out_channels=hidden_channels,
            kernel_size=5,
            padding=2
        )
        self.bn1 = nn.BatchNorm1d(hidden_channels)

        self.conv2 = nn.Conv1d(
            in_channels=hidden_channels,
            out_channels=hidden_channels,
            kernel_size=5,
            padding=2
        )
        self.bn2 = nn.BatchNorm1d(hidden_channels)

        self.lif = neuron.LIFNode(
            v_threshold=0.9,
            surrogate_function=surrogate.ATan(alpha=2.0)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.lif(x)
        return x

In [ ]:
model_stage1 = FeatureExtractionLayer(in_channels=22, hidden_channels=16)

test_output = model_stage1(batch_x)
print("Output shape:", test_output.shape)
print("Spike rate (mean):", test_output.mean().item())
print("Number of non-zero spikes:", (test_output != 0).sum().item(), "out of", test_output.numel())

Output shape: torch.Size([16, 16, 1001])
Spike rate (mean): 0.0362371988594532
Number of non-zero spikes: 9286 out of 256256


In [ ]:
class EEGDecoder(nn.Module):
    def __init__(self, hidden_channels=16):
        super().__init__()

        self.conv1 = nn.Conv1d(hidden_channels, hidden_channels, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(hidden_channels)
        self.lif1 = neuron.LIFNode(v_threshold=0.9, surrogate_function=surrogate.ATan(alpha=2.0))
        self.pool1 = nn.AvgPool1d(kernel_size=2)

        self.conv2 = nn.Conv1d(hidden_channels, hidden_channels, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(hidden_channels)
        self.lif2 = neuron.LIFNode(v_threshold=0.9, surrogate_function=surrogate.ATan(alpha=2.0))
        self.pool2 = nn.AvgPool1d(kernel_size=2)

        self.conv3 = nn.Conv1d(hidden_channels, hidden_channels, kernel_size=5, padding=2)
        self.bn3 = nn.BatchNorm1d(hidden_channels)
        self.lif3 = neuron.LIFNode(v_threshold=0.9, surrogate_function=surrogate.ATan(alpha=2.0))

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.lif1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.lif2(x)
        x = self.pool2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = self.lif3(x)

        return x

In [ ]:
model_stage2 = EEGDecoder(hidden_channels=16)

stage1_output = model_stage1(batch_x)
stage2_output = model_stage2(stage1_output)

print("Stage 1 output shape:", stage1_output.shape)
print("Stage 2 (Decoder) output shape:", stage2_output.shape)
print("Spike rate after Decoder:", stage2_output.mean().item())
print("Non-zero spikes:", (stage2_output != 0).sum().item(), "out of", stage2_output.numel())

Stage 1 output shape: torch.Size([16, 16, 1001])
Stage 2 (Decoder) output shape: torch.Size([16, 16, 250])
Spike rate after Decoder: 0.04696875065565109
Non-zero spikes: 3006 out of 64000


In [24]:
class SpikingClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_classes=4):
        super().__init__()

        self.fc1 = nn.Linear(input_size, hidden_size)
        self.lif1 = neuron.LIFNode(v_threshold=0.9, surrogate_function=surrogate.ATan(alpha=2.0))

        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = x.flatten(start_dim=1)

        x = self.fc1(x)
        x = self.lif1(x)

        x = self.fc2(x)

        return x

In [25]:
model_stage3 = SpikingClassifier(input_size=16*250, hidden_size=128, num_classes=4)

stage3_output = model_stage3(stage2_output)

print("Final output shape:", stage3_output.shape)
print("Sample output (first trial):", stage3_output[0])

Final output shape: torch.Size([16, 4])
Sample output (first trial): tensor([-0.0358, -0.0521, -0.0357, -0.0388], grad_fn=<SelectBackward0>)
